# 2. AML Demo — 176 M transactions

IBM's synthetic [Anti-Money-Laundering dataset](https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml).

Download **HI-Large_Trans.csv** from Kaggle (free account required, ⋮ menu next to the file → Download) — the zip lands in `~/Downloads`.

In [1]:
!mkdir -p data/cache metastore warehouse/aml_hi_lg/transactions
!unzip -n ~/Downloads/HI-Large_Trans.csv.zip -d data/cache

Archive:  /home/will/Downloads/HI-Large_Trans.csv.zip
  inflating: data/cache/HI-Large_Trans.csv  
  inflating: data/cache/__MACOSX/._HI-Large_Trans.csv  


## 2.1 CSV → Parquet

One streaming pass with polars (~2 min): snake_case column names, parse timestamps, write snappy Parquet into the warehouse.

(The CSV has two columns both named `Account`; polars calls the second one `Account_duplicated_0`.)

In [2]:
import os
import polars as pl

if not os.path.exists("warehouse/aml_hi_lg/transactions/transactions.parquet"):

    (
        pl.scan_csv("data/cache/HI-Large_Trans.csv",
                    schema_overrides={"Account": pl.String, "Account_duplicated_0": pl.String,
                                      "Is Laundering": pl.Int32})
        .rename({"Timestamp": "txn_ts", 
                 "From Bank": "from_bank", 
                 "Account": "from_account",
                 "To Bank": "to_bank", 
                 "Account_duplicated_0": "to_account",
                 "Amount Received": "amount_received", 
                 "Receiving Currency": "receiving_currency",
                 "Amount Paid": "amount_paid", 
                 "Payment Currency": "payment_currency",
                 "Payment Format": "payment_format", 
                 "Is Laundering": "is_laundering"})

        .with_columns(txn_ts=pl.col("txn_ts").str.to_datetime("%Y/%m/%d %H:%M", time_unit="us"))
    
        .with_columns(txn_date=pl.col("txn_ts").dt.date())
    
        .select("from_bank", 
                "from_account", 
                "to_bank", 
                "to_account", 
                "amount_received",
                "receiving_currency", 
                "amount_paid", 
                "payment_currency", 
                "payment_format",
                "is_laundering", 
                "txn_ts", 
                "txn_date")

        .sink_parquet("warehouse/aml_hi_lg/transactions/transactions.parquet", compression="snappy")
    )

## 2.2 Create the table

In [3]:
import prestodb

conn = prestodb.dbapi.connect(host="localhost", 
                              port=8080, 
                              user="nb", 
                              catalog="hive", 
                              schema="demo")

cur = conn.cursor()

cur.execute("CREATE SCHEMA IF NOT EXISTS hive.aml_hi_lg")

cur.execute("""CREATE TABLE IF NOT EXISTS hive.aml_hi_lg.transactions 
            (
                from_bank BIGINT, 
                from_account VARCHAR, 
                to_bank BIGINT, 
                to_account VARCHAR,
                amount_received DOUBLE, 
                receiving_currency VARCHAR, 
                amount_paid DOUBLE,
                payment_currency VARCHAR, 
                payment_format VARCHAR, 
                is_laundering INTEGER,
                txn_ts TIMESTAMP, 
                txn_date DATE
            ) WITH (format='PARQUET', external_location='file:///home/warehouse/aml_hi_lg/transactions')
            """)

## 2.3 Query

In [4]:
cur.execute("SELECT count(*) FROM hive.aml_hi_lg.transactions")
cur.fetchall()

[[179702229]]

In [5]:
cur.execute("""SELECT 
                payment_format, 
                count(*) AS txns, 
                round(sum(amount_paid)) AS total_paid
            FROM hive.aml_hi_lg.transactions 
            WHERE is_laundering = 1
            GROUP BY payment_format 
            ORDER BY total_paid DESC
            LIMIT 10
            """)
cur.fetchall()

[['ACH', 199164, 14683984232438.0],
 ['Cheque', 13260, 401413481970.0],
 ['Cash', 3663, 22165756111.0],
 ['Credit Card', 7846, 238103052.0],
 ['Bitcoin', 1608, 936681.0],
 ['Wire', 5, 28671.0]]

In [6]:
import polars as pl

cur.execute("SELECT * FROM hive.aml_hi_lg.transactions LIMIT 10")

pl.DataFrame(
    cur.fetchall(), 
    schema=[d[0] for d in cur.description], 
    orient="row"
    )

In [7]:
cur.close()